In [ ]:
import re

print("Required libraries imported successfully.")

Required libraries imported successfully.


In [ ]:
class Stack:

    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        if self.is_empty():
            return None
        return self.items.pop()

    def peek(self):
        if self.is_empty():
            return None
        return self.items[-1]

    def is_empty(self):
        return len(self.items) == 0

    def display(self):
        return self.items.copy()


print("Custom Stack implementation created.")

Custom Stack implementation created.


In [ ]:
class ArithmeticValidator:

    grammar = """
    E -> T ((+ | -) T)*
    T -> U ((* | /) U)*
    U -> (+ | -) U | P
    P -> A (^ U)?
    A -> NUMBER | IDENTIFIER | ( E )
    """

    def tokenize(self, expression):

        pattern = r'\s*(?:(\d+(?:\.\d+)?)|([a-zA-Z_]\w*)|([+\-*/^()]))'

        tokens = []
        position = 0

        while position < len(expression):

            match = re.match(pattern, expression[position:])

            if not match:
                if expression[position:].strip() == "":
                    break

                raise ValueError("Invalid character")

            number, identifier, operator = match.groups()

            if number:
                tokens.append(("NUMBER", number))

            elif identifier:
                tokens.append(("ID", identifier))

            else:
                tokens.append(("OP", operator))

            position += match.end()

        return tokens

    def validate(self, expression):

        try:
            self.tokens = self.tokenize(expression)
            self.position = 0

            if not self.tokens:
                return False

            self.parse_expression()

            return self.position == len(self.tokens)

        except (ValueError, IndexError):
            return False

    def current(self):

        if self.position < len(self.tokens):
            return self.tokens[self.position]

        return None

    def accept(self, value):

        token = self.current()

        if token and token[1] == value:
            self.position += 1
            return True

        return False

    def parse_expression(self):

        self.parse_term()

        while self.accept("+") or self.accept("-"):
            self.parse_term()

    def parse_term(self):

        self.parse_unary()

        while self.accept("*") or self.accept("/"):
            self.parse_unary()

    def parse_unary(self):

        if self.accept("+") or self.accept("-"):
            self.parse_unary()

        else:
            self.parse_power()

    def parse_power(self):

        self.parse_atom()

        if self.accept("^"):
            self.parse_unary()

    def parse_atom(self):

        token = self.current()

        if token is None:
            raise ValueError("Missing operand")

        if token[0] in ("NUMBER", "ID"):
            self.position += 1

        elif self.accept("("):

            self.parse_expression()

            if not self.accept(")"):
                raise ValueError("Missing closing parenthesis")

        else:
            raise ValueError("Invalid expression")


print("Arithmetic Expression Validator created.")

Arithmetic Expression Validator created.


In [ ]:
arithmetic = ArithmeticValidator()

expressions = [
    "(a+b)*c",
    "a+b*c",
    "(a+b",
    "a++b",
    "a+b)",
    "10/(2+3)",
    "a^b^c"
]

print("ARITHMETIC EXPRESSION VALIDATION")
print("-" * 45)

for expression in expressions:

    result = arithmetic.validate(expression)

    print(expression, "=>",
          "VALID" if result else "INVALID")

ARITHMETIC EXPRESSION VALIDATION
---------------------------------------------
(a+b)*c => VALID
a+b*c => VALID
(a+b => INVALID
a++b => VALID
a+b) => INVALID
10/(2+3) => VALID
a^b^c => VALID


In [ ]:
class SyntaxChecker:

    def tokenize(self, code):

        pattern = re.compile(
            r'\s+|'
            r'\d+(?:\.\d+)?|'
            r'[a-zA-Z_]\w*|'
            r'==|[+\-*/^=()]|[{};]'
        )

        tokens = []
        position = 0

        while position < len(code):

            match = pattern.match(code, position)

            if not match:
                raise ValueError("Invalid character")

            value = match.group()

            if not value.isspace():
                tokens.append(value)

            position = match.end()

        return tokens

    def validate(self, code):

        try:
            self.tokens = self.tokenize(code)
            self.position = 0

            self.parse_program()

            return self.position == len(self.tokens)

        except (ValueError, IndexError):
            return False

    def current(self):

        if self.position < len(self.tokens):
            return self.tokens[self.position]

        return None

    def accept(self, value):

        if self.current() == value:
            self.position += 1
            return True

        return False

    def expect(self, value):

        if not self.accept(value):
            raise ValueError("Expected " + value)

    def parse_program(self):

        while self.current() is not None:

            if self.current() == "}":
                break

            self.parse_statement()

    def parse_statement(self):

        token = self.current()

        if token == "let":
            self.position += 1
            self.parse_assignment()

        elif token == "print":
            self.parse_print()

        elif token == "if":
            self.parse_if()

        elif token and re.fullmatch(r"[a-zA-Z_]\w*", token):
            self.parse_assignment()

        else:
            raise ValueError("Invalid statement")

    def parse_assignment(self):

        identifier = self.current()

        if not identifier or not re.fullmatch(
            r"[a-zA-Z_]\w*", identifier
        ):
            raise ValueError("Invalid variable")

        self.position += 1

        self.expect("=")

        self.parse_expression()

        self.expect(";")

    def parse_print(self):

        self.expect("print")
        self.expect("(")

        self.parse_expression()

        self.expect(")")
        self.expect(";")

    def parse_if(self):

        self.expect("if")
        self.expect("(")

        self.parse_expression()

        self.expect(")")
        self.expect("{")

        self.parse_program()

        self.expect("}")

        if self.accept("else"):

            self.expect("{")

            self.parse_program()

            self.expect("}")

    def parse_expression(self):

        self.parse_term()

        while self.current() in ("+", "-"):

            self.position += 1

            self.parse_term()

    def parse_term(self):

        self.parse_factor()

        while self.current() in ("*", "/"):

            self.position += 1

            self.parse_factor()

    def parse_factor(self):

        if self.accept("+") or self.accept("-"):
            self.parse_factor()
            return

        if self.accept("("):

            self.parse_expression()

            self.expect(")")

            return

        token = self.current()

        if token and (
            re.fullmatch(r"\d+(?:\.\d+)?", token)
            or re.fullmatch(r"[a-zA-Z_]\w*", token)
        ):

            self.position += 1

            return

        raise ValueError("Invalid expression")


print("Programming Language Syntax Checker created.")

Programming Language Syntax Checker created.


In [ ]:
syntax = SyntaxChecker()

programs = [
    "x = 10;",
    "let total = a + b;",
    "print(x);",
    "if (x + 1) { print(x); }",
    "if (a) { if (b) { print(c); } }",
    "x = a + ;",
    "print(x;",
    "if (x) { print(x);"
]

print("PROGRAMMING LANGUAGE SYNTAX CHECKER")
print("-" * 50)

for program in programs:

    result = syntax.validate(program)

    print(program, "=>",
          "VALID" if result else "INVALID")

PROGRAMMING LANGUAGE SYNTAX CHECKER
--------------------------------------------------
x = 10; => VALID
let total = a + b; => VALID
print(x); => VALID
if (x + 1) { print(x); } => VALID
if (a) { if (b) { print(c); } } => VALID
x = a + ; => INVALID
print(x; => INVALID
if (x) { print(x); => INVALID


In [ ]:
class ParenthesesChecker:

    opening = {
        "(": ")",
        "{": "}",
        "[": "]"
    }

    closing = [")", "}", "]"]

    def validate(self, expression):

        stack = Stack()

        for character in expression:

            if character in self.opening:

                stack.push(character)

            elif character in self.closing:

                if stack.is_empty():
                    return False

                top = stack.pop()

                if self.opening[top] != character:
                    return False

        return stack.is_empty()


print("Balanced Parentheses Checker created.")

Balanced Parentheses Checker created.


In [ ]:
parentheses = ParenthesesChecker()

brackets = [
    "()",
    "{[()]}",
    "((()))",
    "([)]",
    "(((",
    "())"
]

print("BALANCED PARENTHESES CHECKER")
print("-" * 45)

for expression in brackets:

    result = parentheses.validate(expression)

    print(expression, "=>",
          "BALANCED" if result else "NOT BALANCED")

BALANCED PARENTHESES CHECKER
---------------------------------------------
() => BALANCED
{[()]} => BALANCED
((())) => BALANCED
([)] => NOT BALANCED
((( => NOT BALANCED
()) => NOT BALANCED


In [ ]:
class PalindromeChecker:

    def validate(self, string):

        if not re.fullmatch(r"[ab]*", string):
            return False

        length = len(string)

        # Try every possible midpoint.
        # This simulates nondeterministic PDA choices.

        for midpoint in range(length + 1):

            stack = Stack()

            # Push the first half.
            for character in string[:midpoint]:
                stack.push(character)

            # Skip the middle symbol for odd lengths.
            remaining_start = midpoint

            if length % 2 == 1:
                if midpoint <= length - 1:
                    remaining_start = midpoint + 1

            matched = True

            # Compare the second half with the stack.
            for character in string[remaining_start:]:

                if stack.is_empty():
                    matched = False
                    break

                if stack.pop() != character:
                    matched = False
                    break

            if matched and stack.is_empty():
                return True

        return False


print("Palindrome Checker created.")

Palindrome Checker created.


In [ ]:
palindrome = PalindromeChecker()

strings = [
    "abba",
    "aba",
    "a",
    "ab",
    "baab",
    "aab",
    ""
]

print("PALINDROME CHECKER")
print("-" * 40)

for string in strings:

    result = palindrome.validate(string)

    print(repr(string), "=>",
          "PALINDROME" if result else "NOT PALINDROME")

PALINDROME CHECKER
----------------------------------------
'abba' => PALINDROME
'aba' => PALINDROME
'a' => PALINDROME
'ab' => NOT PALINDROME
'baab' => PALINDROME
'aab' => NOT PALINDROME
'' => PALINDROME


In [ ]:
class PasswordValidator:

    pattern = re.compile(
        r"(?=.*[A-Z])"
        r"(?=.*[a-z])"
        r"(?=.*[0-9])"
        r"(?=.*[^a-zA-Z0-9\s])"
        r"\S{12,64}"
    )

    def validate(self, password):

        return bool(self.pattern.fullmatch(password))


print("Password Pattern Validator created.")

Password Pattern Validator created.


In [ ]:
password = PasswordValidator()

passwords = [
    "StrongPass9!",
    "weak",
    "StrongPassword123!",
    "password123",
    "NoSpecialChar123",
    "NoNumberHere!!"
]

print("PASSWORD PATTERN VALIDATION")
print("-" * 45)

for value in passwords:

    result = password.validate(value)

    print(
        "Length:", len(value),
        "=>",
        "VALID FORMAT" if result else "INVALID FORMAT"
    )

PASSWORD PATTERN VALIDATION
---------------------------------------------
Length: 12 => VALID FORMAT
Length: 4 => INVALID FORMAT
Length: 18 => VALID FORMAT
Length: 11 => INVALID FORMAT
Length: 16 => INVALID FORMAT
Length: 14 => INVALID FORMAT


In [ ]:
def run_all_tests():

    arithmetic = ArithmeticValidator()
    syntax = SyntaxChecker()
    parentheses = ParenthesesChecker()
    palindrome = PalindromeChecker()
    password = PasswordValidator()

    print("\n========== ALL CASE STUDIES ==========")

    print("\n1. ARITHMETIC EXPRESSION")

    for value in ["(a+b)*c", "a++b", "a+b*c"]:
        print(value, "=>", arithmetic.validate(value))

    print("\n2. PROGRAMMING SYNTAX")

    for value in ["x = 10;", "print(x);", "print(x;"]:
        print(value, "=>", syntax.validate(value))

    print("\n3. BALANCED PARENTHESES")

    for value in ["()", "{[()]}", "([)]"]:
        print(value, "=>", parentheses.validate(value))

    print("\n4. PALINDROME")

    for value in ["abba", "aba", "aab"]:
        print(value, "=>", palindrome.validate(value))

    print("\n5. PASSWORD FORMAT")

    for value in ["StrongPass9!", "weak"]:
        print(
            len(value),
            "characters =>",
            password.validate(value)
        )


run_all_tests()


========== ALL CASE STUDIES ==========

1. ARITHMETIC EXPRESSION
(a+b)*c => True
a++b => True
a+b*c => True

2. PROGRAMMING SYNTAX
x = 10; => True
print(x); => True
print(x; => False

3. BALANCED PARENTHESES
() => True
{[()]} => True
([)] => False

4. PALINDROME
abba => True
aba => True
aab => False

5. PASSWORD FORMAT
12 characters => True
4 characters => False


In [ ]:
print("PDA AND CFG CASE STUDY")
print("1. Arithmetic Expression Validation")
print("2. Programming Language Syntax Checker")
print("3. Balanced Parentheses Checker")
print("4. Palindrome Checker")
print("5. Password Pattern Validation")
print("6. Run All Test Cases")

choice = input("Enter your choice: ")

if choice == "1":
    value = input("Enter expression: ")
    print("VALID" if arithmetic.validate(value) else "INVALID")

elif choice == "2":
    value = input("Enter programming statement: ")
    print("VALID" if syntax.validate(value) else "INVALID")

elif choice == "3":
    value = input("Enter brackets: ")
    print("BALANCED" if parentheses.validate(value) else "NOT BALANCED")

elif choice == "4":
    value = input("Enter string: ")
    print("PALINDROME" if palindrome.validate(value) else "NOT PALINDROME")

elif choice == "5":
    value = input("Enter password: ")
    print("VALID FORMAT" if password.validate(value) else "INVALID FORMAT")

elif choice == "6":
    run_all_tests()

else:
    print("Invalid choice")

PDA AND CFG CASE STUDY
1. Arithmetic Expression Validation
2. Programming Language Syntax Checker
3. Balanced Parentheses Checker
4. Palindrome Checker
5. Password Pattern Validation
6. Run All Test Cases
Enter your choice: 1
Enter expression: (a+b)*c
VALID
